# Web Research Agent Exploration

This notebook explains the web research agent in small, runnable steps. Run the cells from top to bottom. The search and synthesis cells call Tavily and OpenAI, so valid API keys are required.

## 1. Load dependencies and environment

The production script loads `.env` values before creating the Tavily and OpenAI clients.

In [2]:
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

load_dotenv()

print("Dependencies imported successfully.")

Dependencies imported successfully.


In [3]:
# Check configuration without displaying secret values.
required_keys = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
configuration_status = {
    key: "configured" if os.getenv(key) else "missing"
    for key in required_keys
}
configuration_status

{'OPENAI_API_KEY': 'configured', 'TAVILY_API_KEY': 'configured'}

## 2. Define the shared research state

LangGraph passes this state from the search node to the synthesis node.

In [4]:
class ResearchState(TypedDict):
    """State shared by the search and synthesis stages."""

    messages: Annotated[list, add_messages]
    query: str
    search_results: list[dict]
    report: str


query = "latest trends in AI and agentic AI"
state = {
    "query": query,
    "messages": [],
    "search_results": [],
    "report": "",
}

state

{'query': 'latest trends in AI and agentic AI',
 'messages': [],
 'search_results': [],
 'report': ''}

## 3. Search the web

This cell mirrors `search_web` from `agent.py`. It makes the external Tavily call and stores the raw response for inspection.

In [5]:
search_tool = TavilySearch(max_results=5)
raw_results = search_tool.invoke(query)

print(f"Raw response type: {type(raw_results).__name__}")
raw_results

Raw response type: dict


{'query': 'latest trends in AI and agentic AI',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://aimultiple.com/agentic-ai-trends',
   'title': '10+ Agentic AI Trends and Examples',
   'content': 'Agentic AI trends with real-life examples\n\n1. Towards autonomous, self-healing data pipelines\n\n2. Tooling over process\n\n3. Vertical AI agents in specialized industries\n\n4. Integration of AI agents with the physical world\n\n5. Growing shift towards open-source models\n\n6. Transformative artificial intelligence\n\n7. AI agent building frameworks\n\n8. Combining synthetic and real-world data\n\n9. Agentic AI reshaping team roles\n\n10. The human element in agentic AI [...] Hazal Şimşek\n\nupdated onAug 17, 2026\n\nSee ourethical norms\n\nCite This Research\n\nWe reviewed and compared Agentic AI trends from several major industry reports, benchmarks, and vendor disclosures. The sources point out that the future of agentic AI is about integratin

In [6]:
# Normalize Tavily's possible response shapes to a list of dictionaries.
if isinstance(raw_results, dict):
    search_results = raw_results.get("results", [])
elif isinstance(raw_results, list):
    search_results = raw_results
else:
    search_results = []

state["search_results"] = search_results

print(f"Normalized results: {len(search_results)}")
for index, result in enumerate(search_results, start=1):
    print(f"{index}. {result.get('title', 'Untitled')} - {result.get('url', 'N/A')}")

Normalized results: 5
1. 10+ Agentic AI Trends and Examples - https://aimultiple.com/agentic-ai-trends
2. The 7 Agentic AI Trends Shaping Enterprise Supply Chains in 2026 - https://finance.yahoo.com/news/7-agentic-ai-trends-shaping-133500708.html
3. Agentic AI - 2026 Market & Investments Trends - https://tracxn.com/d/sectors/agentic-ai/__oyRAfdUfHPjf2oap110Wis0Qg12Gd8DzULlDXPJzrzs
4. Agentic AI in DACH - 2026 Market & Investments Trends - Tracxn - https://tracxn.com/d/explore/agentic-ai-startups-in-dach/__NqYXQKSwbGhafqBWMcpVno17dRB1sTzw0sytWpoFnFU
5. The agentic reality check: Preparing for a silicon-based workforce - https://www.deloitte.com/us/en/insights/topics/technology-management/tech-trends/2026/agentic-ai-strategy.html


## 4. Build the synthesis prompt

The production agent keeps source excerpts short so the model receives focused evidence.

In [7]:
results_text = "\n\n".join(
    f"Source: {result.get('url', 'N/A')}\n"
    f"Title: {result.get('title', 'N/A')}\n"
    f"Content: {result.get('content', '')[:500]}"
    for result in search_results
)

messages = [
    SystemMessage(
        content=(
            "You are a research analyst. Synthesize the search results into "
            "a clear, structured report with: Summary, Key Findings "
            "(bullet points), and Sources."
        )
    ),
    HumanMessage(
        content=f"Research query: {query}\n\nSearch results:\n{results_text}"
    ),
]

print(f"Prepared {len(messages)} messages for synthesis.")
print(messages[1].content[:1000])

Prepared 2 messages for synthesis.
Research query: latest trends in AI and agentic AI

Search results:
Source: https://aimultiple.com/agentic-ai-trends
Title: 10+ Agentic AI Trends and Examples
Content: Agentic AI trends with real-life examples

1. Towards autonomous, self-healing data pipelines

2. Tooling over process

3. Vertical AI agents in specialized industries

4. Integration of AI agents with the physical world

5. Growing shift towards open-source models

6. Transformative artificial intelligence

7. AI agent building frameworks

8. Combining synthetic and real-world data

9. Agentic AI reshaping team roles

10. The human element in agentic AI [...] Hazal Şimşek

updated onAug 17, 202

Source: https://finance.yahoo.com/news/7-agentic-ai-trends-shaping-133500708.html
Title: The 7 Agentic AI Trends Shaping Enterprise Supply Chains in 2026
Content: 2 min read

TORONTO, Feb. 3, 2026 /PRNewswire/ -- Agentic AI has moved rapidly from experimentation to enterprise deployment, partic

## 5. Generate a report directly

This optional cell calls the model without LangGraph so the synthesis step can be understood in isolation.

In [8]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response = llm.invoke(messages)

report = response.content
print(report)

# Report on Latest Trends in AI and Agentic AI

## Summary
The landscape of artificial intelligence (AI) is rapidly evolving, particularly in the realm of agentic AI, which refers to AI systems capable of autonomous decision-making and actions. Recent trends indicate a significant shift towards the integration of AI in various sectors, especially in supply chain management, where efficiency and accuracy are paramount. This report synthesizes the latest findings on agentic AI trends, highlighting key developments and market dynamics.

## Key Findings
- **Autonomous Data Pipelines**: There is a growing trend towards the development of self-healing data pipelines that can autonomously manage and rectify data issues.
- **Tooling Over Process**: Emphasis is shifting from rigid processes to flexible tooling that allows for more adaptive AI solutions.
- **Vertical AI Agents**: Specialized AI agents are emerging in various industries, tailored to meet specific operational needs.
- **Integratio

## 6. Rebuild the LangGraph workflow

This graph is the same sequence used by the CLI and Streamlit application: search, then synthesize, then finish.

In [9]:
def search_web(state: ResearchState) -> ResearchState:
    """Search Tavily and normalize the response into the graph state."""

    tool = TavilySearch(max_results=5)
    raw_results = tool.invoke(state["query"])

    if isinstance(raw_results, dict):
        results = raw_results.get("results", [])
    elif isinstance(raw_results, list):
        results = raw_results
    else:
        results = []

    return {"search_results": results}


def synthesize_report(state: ResearchState) -> ResearchState:
    """Create a report from the search results using the chat model."""

    results_text = "\n\n".join(
        f"Source: {result.get('url', 'N/A')}\n"
        f"Title: {result.get('title', 'N/A')}\n"
        f"Content: {result.get('content', '')[:500]}"
        for result in state["search_results"]
    )
    messages = [
        SystemMessage(
            content=(
                "You are a research analyst. Synthesize the search results "
                "into a clear, structured report with: Summary, Key Findings "
                "(bullet points), and Sources."
            )
        ),
        HumanMessage(
            content=f"Research query: {state['query']}\n\nSearch results:\n{results_text}"
        ),
    ]
    response = ChatOpenAI(model="gpt-4o-mini", temperature=0).invoke(messages)
    return {"report": response.content, "messages": [response]}

In [10]:
graph = StateGraph(ResearchState)
graph.add_node("search", search_web)
graph.add_node("synthesize", synthesize_report)
graph.set_entry_point("search")
graph.add_edge("search", "synthesize")
graph.add_edge("synthesize", END)

agent = graph.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


## 7. Run the complete agent

This final cell invokes the compiled graph with the same initial state used by `agent.py` and returns the report plus search results.

In [11]:
final_state = agent.invoke(
    {
        "query": query,
        "messages": [],
        "search_results": [],
        "report": "",
    }
)

print(final_state["report"])
print(f"\nSources returned: {len(final_state['search_results'])}")

# Report on Latest Trends in AI and Agentic AI

## Summary
The landscape of artificial intelligence (AI) is rapidly evolving, with a particular focus on agentic AI, which refers to AI systems capable of making autonomous decisions and taking actions based on their environment. This report synthesizes the latest trends in agentic AI, highlighting its implications for automation, business transformation, and technological advancements.

## Key Findings
- **Hyperautomation Expansion**: Agentic AI is enhancing hyperautomation by enabling systems to make informed, autonomous decisions, moving beyond simple task automation to complex decision-making processes.
  
- **Autonomous Data Pipelines**: There is a trend towards developing self-healing data pipelines that can autonomously manage and rectify issues without human intervention.

- **Vertical AI Agents**: Specialized AI agents are emerging in various industries, tailored to meet specific sector needs and challenges.

- **Integration with